In [1]:
"""
After manual analysis, I've found some mistakes happening in the DB results.
In this notebook, I pick the incorrect records to test the DB_1record_evaluator, seeing if it successfully tells that these records are incorrect.
Extraction of text_doc and file_path_doc is also done here, so that it can be done once for 1 notegroup instead of repeatedly done for each record
"""
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from oral_notes.evaluate.DB_1record_evaluator import DB1recordEvaluator
import sqlite3

pipeline_type="baseline_v2"
DB_PATH = "DB/oedb_baseline_v2.db"
schema_path = "data/metadata_DB/schema_v2.yaml"
prompt_path_evaluator = "data/prompt_templates/prompt_evaluator.yaml"
service_account_file="config/service_account_key.json"

def textdoc_extractor(notegroup_id, DB_PATH, service_account_file):
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()
        if row is None:
            raise ValueError(f"No notegroup found with ID {notegroup_id}")
        note_url_qa, note_url_participant = row

    file_loader = GoogleDriveLoader(service_account_file)
    extractor = TextExtractor()

    all_texts = {}
    all_drive_paths = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            print(f"\n--- Skipping {label}: no URL ---")
            continue
        print(f"\n--- Loading and extracting text from {label} ---")
        result = file_loader.load(url)
        text = extractor.extract(result)
        all_texts[label] = f"[Data source: {result['name']}]\n{text}"
        all_drive_paths[label] = result['drive_path']
    combined_drive_paths = "|".join(all_drive_paths.values())
    combined_text = "\n---\n".join(
        t for t in [all_texts.get('PARTICIPANT', ''), all_texts.get('QA', '')] if t
    )
    return combined_text, combined_drive_paths

In [2]:
evaluator_version="initial_test"
notegroupID=3
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
# run the evaluator over the target records
records_to_evaluate = (
    [("notegroups", 3)]
#    + [("participants", pk) for pk in range(14, 19)]   # 14-18
#    + [("questions", pk) for pk in range(44, 49)]      # 44-48
)

for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)


2026-06-16 09:56:23 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=3:
{
  "date": "2024-11-22",
  "data_source_category": "expertpool"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling

--- Evaluating notegroups pk=3 ---


2026-06-16 09:56:24 | INFO     | utils.token_logger | Token usage [notegroups pk=3] attempt 1 — in: 6654 (cached: 6528), out: 25, cost: $0.001223
2026-06-16 09:56:24 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=3] correct=0 wrong_fields=['date']


In [3]:
# run the evaluator over the target records
records_to_evaluate = (
#    [("notegroups", 3)]
    [("participants", pk) for pk in range(14, 19)]   # 14-18
    + [("questions", pk) for pk in range(44, 49)]      # 44-48
)

for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")

2026-06-16 09:59:01 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=14:
{
  "full_name": "Mortada Abu Hassan"
}



--- Evaluating participants pk=14 ---


2026-06-16 09:59:03 | INFO     | utils.token_logger | Token usage [participants pk=14] attempt 1 — in: 7242 (cached: 6272), out: 26, cost: $0.002256
2026-06-16 09:59:03 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=14] correct=0 wrong_fields=['full_name']
2026-06-16 09:59:03 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=15:
{
  "full_name": "Ahmad Alhussein Alsatouf"
}



--- Evaluating participants pk=15 ---


2026-06-16 09:59:04 | INFO     | utils.token_logger | Token usage [participants pk=15] attempt 1 — in: 7246 (cached: 6272), out: 23, cost: $0.002231
2026-06-16 09:59:04 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=15] correct=1 wrong_fields=[]
2026-06-16 09:59:04 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=16:
{
  "full_name": "Alaa Abdal Wahab"
}


...

2026-06-16 09:59:11 | INFO     | utils.token_logger | Token usage [questions pk=45] attempt 1 — in: 7338 (cached: 6400), out: 23, cost: $0.002203
2026-06-16 09:59:11 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=45] correct=1 wrong_fields=[]
2026-06-16 09:59:12 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=47:
{
  "question_content": "● Does the process of obtaining a Citizen Service Number (BSN) affect your ability to find work? If so, how?",
  "main_indicator": [
    "work",
    "rights and responsibilities",
    "werk & inkomen",
    "rechten & verantwoordelijkheden"
  ]
}



--- Evaluating questions pk=46 ---
  FAILED: No record found in 'questions' for questionID=46

--- Evaluating questions pk=47 ---


2026-06-16 09:59:12 | INFO     | utils.token_logger | Token usage [questions pk=47] attempt 1 — in: 7309 (cached: 6400), out: 23, cost: $0.002166
2026-06-16 09:59:12 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=47] correct=1 wrong_fields=[]
2026-06-16 09:59:13 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=48:
{
  "question_content": "● Has the uncertainty/waiting period in the asylum seekers' center affected your motivation or chances of finding work? If so, how?",
  "main_indicator": [
    "work",
    "stability",
    "werk & inkomen",
    "stabilitieit"
  ]
}



--- Evaluating questions pk=48 ---


2026-06-16 09:59:14 | INFO     | utils.token_logger | Token usage [questions pk=48] attempt 1 — in: 7310 (cached: 6400), out: 23, cost: $0.002168
2026-06-16 09:59:14 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=48] correct=1 wrong_fields=[]


In [4]:
evaluator_version="initial_test"
notegroupID=7
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("notegroups", 7)]
    + [("participants", pk) for pk in range(28, 32)]   # 28-31
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-16 12:49:28 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=7:
{
  "date": "2024-11-28",
  "data_source_category": "expertpool"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating notegroups pk=7 ---


2026-06-16 12:49:29 | INFO     | utils.token_logger | Token usage [notegroups pk=7] attempt 1 — in: 7974 (cached: 0), out: 25, cost: $0.010218
2026-06-16 12:49:29 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=7] correct=0 wrong_fields=['date']
2026-06-16 12:49:29 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=28:
{
  "full_name": "Rawa Alshumry",
  "initials": "R"
}



--- Evaluating participants pk=28 ---


...

2026-06-16 12:49:33 | INFO     | utils.token_logger | Token usage [participants pk=29] attempt 1 — in: 8570 (cached: 7680), out: 29, cost: $0.002363
2026-06-16 12:49:33 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=29] correct=0 wrong_fields=['municipality', 'initials']
2026-06-16 12:49:33 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=30:
{
  "full_name": "Abdullah Najjar",
  "initials": "A.N",
  "language_group": [
    "English",
    "Arabic",
    "Turkish"
  ],
  "municipality": "Utrecht"
}



--- Evaluating participants pk=30 ---


2026-06-16 12:49:34 | INFO     | utils.token_logger | Token usage [participants pk=30] attempt 1 — in: 8588 (cached: 0), out: 29, cost: $0.011025
2026-06-16 12:49:34 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=30] correct=0 wrong_fields=['language_group', 'municipality']
2026-06-16 12:49:34 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=31:
{
  "full_name": "Victor",
  "initials": "V",
  "language_group": [
    "English",
    "Arabic"
  ],
  "first_arrival_date": "2021-11-11",
  "municipality": "Rotterdam",
  "age": 41
}



--- Evaluating participants pk=31 ---


2026-06-16 12:49:35 | INFO     | utils.token_logger | Token usage [participants pk=31] attempt 1 — in: 8599 (cached: 7680), out: 33, cost: $0.002439
2026-06-16 12:49:35 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=31] correct=0 wrong_fields=['language_group', 'first_arrival_date', 'age']


In [2]:
evaluator_version="initial_test"
notegroupID=22
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("notegroups", 22)]
    + [("participants", pk) for pk in range(70, 75)]
    + [("questions", pk) for pk in [*range(421, 423), *range(427, 431)]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Notites_Mahad_2.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/Notites_Mahad_2.docx

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-17 17:27:53 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=22:
{}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating notegroups pk=22 ---


2026-06-17 17:27:55 | INFO     | utils.token_logger | Token usage [notegroups pk=22] attempt 1 — in: 9922 (cached: 0), out: 29, cost: $0.012693
2026-06-17 17:27:55 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=22] correct=0 wrong_fields=['date', 'data_source_category']
2026-06-17 17:27:55 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=70:
{
  "full_name": "Mohamed Hardan",
  "initials": "Mohamed 1",
  "gender": "Man / Man / رجل / Erkek",
  "learning_route": "Z-route",
  "place_of_origin": "Syrië",
  "language_group": [
    "Arabisch"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Monnickendam",
  "age": 45
}



--- Evaluating participants pk=70 ---


...

2026-06-17 17:28:05 | INFO     | utils.token_logger | Token usage [questions pk=428] attempt 1 — in: 10592 (cached: 9472), out: 23, cost: $0.002814
2026-06-17 17:28:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=428] correct=1 wrong_fields=[]
2026-06-17 17:28:05 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=429:
{
  "question_content": "Voor ons betekent zelfstandigheid dat je je eigen administratie kunt regelen zonder hulp zoals brieven begrijpen, formulieren invullen en afspraken maken met instanties.",
  "main_indicator": [
    "stability",
    "rights and responsibilities"
  ],
  "followed_questionID": 427,
  "followed_question_content": "Sommige dingen kunnen we inmiddels zelf regelen, zoals eenvoudige dagelijkse zaken. Maar bij officiële brieven, formulieren en afspraken hebben we vaak nog hulp nodig. Dit komt vooral doordat de taal soms moeilijk is en omdat de regels in Nederland ingewikkeld kunnen 


--- Evaluating questions pk=429 ---


2026-06-17 17:28:06 | INFO     | utils.token_logger | Token usage [questions pk=429] attempt 1 — in: 10564 (cached: 9472), out: 23, cost: $0.002779
2026-06-17 17:28:06 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=429] correct=1 wrong_fields=[]
2026-06-17 17:28:06 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=430:
{
  "question_content": "In het begin van onze tijd in Nederland vonden we de administratie erg stressvol. We kregen veel brieven en formulieren die we niet altijd goed begrepen. Alles was nieuw: de taal, de regels en de instanties. Daardoor waren we vaak bang om fouten te maken of belangrijke afspraken te missen. Langzaam gaat het beter, maar die eerste periode zorgde voor veel spanning en onzekerheid.",
  "main_indicator": [
    "stability",
    "safety"
  ]
}



--- Evaluating questions pk=430 ---


2026-06-17 17:28:07 | INFO     | utils.token_logger | Token usage [questions pk=430] attempt 1 — in: 10504 (cached: 9472), out: 26, cost: $0.002734
2026-06-17 17:28:07 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=430] correct=0 wrong_fields=['question_content']


In [2]:
evaluator_version="initial_test"
notegroupID=14
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 832)]
    + [("participants", pk) for pk in range(49, 53)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Zorgcafe#1_Venlo_notes.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)


2026-06-17 17:32:38 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=832:
{
  "questionID": 313,
  "participantID": 49,
  "answer_content_oriLAN": "The older place was busier, but the new place, for two weeks, for only one client, but the new place is not a good place, it's not busy, so not a lot of people come, it is not very accessible\nAnother place is needed\nmore promotion of the activities\nFinancial support is needed for more motivation- incentive for people to work and also for visitors to join the activities\ngood connection with other organizations, gementee, zorg sistem, regowerken, refugee— samenwerking met andere organities— the connection with the refugee team is good, but with the rest of the organizations not",
  "initials": "S",
  "gender": "Man",
  "place_of_origin": "Iraqi",
  "municipality": "Venlo",
  "question_content": "What can be better?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Zorgcafe#1_Venlo_notes.docx

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=832 ---


2026-06-17 17:32:39 | INFO     | utils.token_logger | Token usage [answers pk=832] attempt 1 — in: 2798 (cached: 0), out: 23, cost: $0.003728
2026-06-17 17:32:39 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=832] correct=1 wrong_fields=[]
2026-06-17 17:32:39 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=49:
{
  "initials": "S",
  "gender": "Man",
  "place_of_origin": "Iraqi",
  "municipality": "Venlo"
}



--- Evaluating participants pk=49 ---


...

2026-06-17 17:32:43 | INFO     | utils.token_logger | Token usage [participants pk=50] attempt 1 — in: 3102 (cached: 2176), out: 23, cost: $0.001659
2026-06-17 17:32:43 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=50] correct=1 wrong_fields=[]
2026-06-17 17:32:43 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=51:
{
  "initials": "T",
  "gender": "Man",
  "place_of_origin": "Turkish",
  "municipality": "Venlo"
}



--- Evaluating participants pk=51 ---


2026-06-17 17:32:44 | INFO     | utils.token_logger | Token usage [participants pk=51] attempt 1 — in: 3103 (cached: 2176), out: 27, cost: $0.001701
2026-06-17 17:32:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=51] correct=0 wrong_fields=['place_of_origin']
2026-06-17 17:32:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=52:
{
  "initials": "P",
  "gender": "Woman",
  "place_of_origin": "Iraqi",
  "municipality": "Venlo"
}



--- Evaluating participants pk=52 ---


2026-06-17 17:32:45 | INFO     | utils.token_logger | Token usage [participants pk=52] attempt 1 — in: 3104 (cached: 2176), out: 23, cost: $0.001662
2026-06-17 17:32:45 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=52] correct=1 wrong_fields=[]


In [3]:
evaluator_version="initial_test"
notegroupID=12
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in range(46, 49)]
    + [("questions", pk) for pk in range(271, 303)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)


2026-06-17 17:50:29 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=46:
{
  "gender": "woman",
  "place_of_origin": "Iraq",
  "municipality": "Den Helder",
  "age": 19
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)

--- Skipping PARTICIPANT: no URL ---

--- Evaluating participants pk=46 ---


2026-06-17 17:50:31 | INFO     | utils.token_logger | Token usage [participants pk=46] attempt 1 — in: 3616 (cached: 0), out: 39, cost: $0.004910
2026-06-17 17:50:31 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=46] correct=0 wrong_fields=['age', 'first_arrival_date', 'participant_group', 'other_information', 'initials']
2026-06-17 17:50:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=47:
{
  "gender": "woman",
  "place_of_origin": "Somalia",
  "municipality": "Den Helder",
  "age": 17
}



--- Evaluating participants pk=47 ---


...

2026-06-17 17:51:05 | INFO     | utils.token_logger | Token usage [questions pk=300] attempt 1 — in: 3643 (cached: 3584), out: 26, cost: $0.000782
2026-06-17 17:51:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=300] correct=0 wrong_fields=['main_indicator']
2026-06-17 17:51:05 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=301:
{
  "question_content": "What do you need to find work?",
  "main_indicator": [
    "work"
  ]
}



--- Evaluating questions pk=301 ---


2026-06-17 17:51:06 | INFO     | utils.token_logger | Token usage [questions pk=301] attempt 1 — in: 3643 (cached: 2688), out: 23, cost: $0.001760
2026-06-17 17:51:06 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=301] correct=1 wrong_fields=[]
2026-06-17 17:51:06 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=302:
{
  "question_content": "How can the municipality help you to find a job?",
  "main_indicator": [
    "work"
  ]
}



--- Evaluating questions pk=302 ---


2026-06-17 17:51:07 | INFO     | utils.token_logger | Token usage [questions pk=302] attempt 1 — in: 3646 (cached: 2688), out: 23, cost: $0.001763
2026-06-17 17:51:07 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=302] correct=1 wrong_fields=[]


In [ ]:
evaluator_version="initial_test"
notegroupID=18
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", 62)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")

In [5]:
evaluator_version="initial_test"
notegroupID=5
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 403)]
    + [("questions", pk) for pk in range(99, 103)]
    + [("participants", pk) for pk in range(25, 27)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Naya notes (application/vnd.google-apps.document)


2026-06-17 20:03:43 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=403:
{
  "questionID": 107,
  "participantID": 26,
  "answer_content_oriLAN": "They embarrassed us in the work process, once they hear we are refugees, they kick us out. There is only one difference between refugees and others, and this shouldnt be taken into consideration, because i have the experience and the TWV.\nAdeel: Racist people here.",
  "full_name": "Arsalan",
  "initials": "A.",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English",
    "Dutch",
    "Pakistani"
  ],
  "municipality": "Zaandam",
  "question_content": "When did the support not work well? What was missing in your opinion? What would you have preferred to see done differently?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=403 ---


2026-06-17 20:03:44 | INFO     | utils.token_logger | Token usage [answers pk=403] attempt 1 — in: 6589 (cached: 0), out: 23, cost: $0.008466
2026-06-17 20:03:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=403] correct=1 wrong_fields=[]
2026-06-17 20:03:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=99:
{
  "question_content": "Do your living conditions in the asylum seekers' center affect your ability to find work or to work?",
  "main_indicator": [
    "work",
    "housing"
  ]
}



--- Evaluating questions pk=99 ---


...

2026-06-17 20:03:49 | INFO     | utils.token_logger | Token usage [questions pk=102] attempt 1 — in: 6967 (cached: 6016), out: 23, cost: $0.002171
2026-06-17 20:03:49 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=102] correct=1 wrong_fields=[]
2026-06-17 20:03:49 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=25:
{
  "full_name": "Adeel",
  "initials": "A.",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English",
    "Dutch",
    "Pakistani"
  ],
  "municipality": "Zaandam"
}



--- Evaluating participants pk=25 ---


2026-06-17 20:03:50 | INFO     | utils.token_logger | Token usage [participants pk=25] attempt 1 — in: 6942 (cached: 5888), out: 36, cost: $0.002414
2026-06-17 20:03:50 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=25] correct=0 wrong_fields=['initials', 'language_group', 'municipality', 'gender', 'status']
2026-06-17 20:03:50 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=26:
{
  "full_name": "Arsalan",
  "initials": "A.",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English",
    "Dutch",
    "Pakistani"
  ],
  "municipality": "Zaandam"
}



--- Evaluating participants pk=26 ---


2026-06-17 20:03:51 | INFO     | utils.token_logger | Token usage [participants pk=26] attempt 1 — in: 6943 (cached: 5888), out: 32, cost: $0.002375
2026-06-17 20:03:51 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=26] correct=0 wrong_fields=['initials', 'language_group', 'municipality']


In [6]:
evaluator_version="initial_test"
notegroupID=5
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 401)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Naya notes (application/vnd.google-apps.document)


2026-06-17 20:06:34 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=401:
{
  "questionID": 106,
  "participantID": 25,
  "answer_content_oriLAN": "Adeel & Arsalan = the refugee is not the issue! We are all here for some reason - they need to equalize the rules! Between work visas and refugee visas! We cannot call ourselves muslims in pakistan! There is a threat to our lives there.",
  "full_name": "Adeel",
  "initials": "A.",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English",
    "Dutch",
    "Pakistani"
  ],
  "municipality": "Zaandam",
  "question_content": "What is needed to improve this?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=401 ---


2026-06-17 20:06:35 | INFO     | utils.token_logger | Token usage [answers pk=401] attempt 1 — in: 6566 (cached: 5888), out: 23, cost: $0.001813
2026-06-17 20:06:35 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=401] correct=1 wrong_fields=[]


In [7]:
evaluator_version="initial_test"
notegroupID=11
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", pk) for pk in range(730, 744)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)


2026-06-17 20:14:17 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=731:
{
  "questionID": 264,
  "participantID": 44,
  "answer_content_oriLAN": "\"Een mens heeft een eigen huis nodig. Leven in het kamp is erg moeilijk – en met kinderen is het nog veel zwaarder.\nS, Turkse, Asielzoeker, Vrouw:\"Het kamp was vroeger een gevangenis – dat merk je aan alles. Het imago is heel negatief, en daardoor kijkt de bevolking in E, Turkse, Asielzoeker, Vrouw:Den Helder met afstand naar ons.\"\n\"Ik mis mijn huis enorm. Leven in het kamp is zwaar. Kunt u alstublieft de fysieke omstandigheden en de hygiëne verbeteren?\"",
  "initials": "S",
  "gender": "Vrouw",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Turkse",
  "municipality": "Den Helder",
  "question_content": "Wat gaat er goed en welke obstakels ervaar je?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=730 ---
  FAILED: 'participantID'

--- Evaluating answers pk=731 ---


2026-06-17 20:14:18 | INFO     | utils.token_logger | Token usage [answers pk=731] attempt 1 — in: 7838 (cached: 0), out: 23, cost: $0.010027
2026-06-17 20:14:18 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=731] correct=1 wrong_fields=[]
2026-06-17 20:14:18 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=732:
{
  "questionID": 265,
  "participantID": 45,
  "answer_content_oriLAN": "Om zelfs tijdelijk te kunnen werken, moet je de taal leren. Gemeenten of maatschappelijke organisaties zouden daar al in het kamp op moeten inzetten – niet via vrijwilligers, maar via professionele taaltrainers.",
  "initials": "S",
  "gender": "Man",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Turkse",
  "municipality": "Den Helder",
  "question_content": "Welke acties onderneem je om je doel te bereiken?"
}



--- Evaluating answers pk=732 ---


...

2026-06-17 20:14:30 | INFO     | utils.token_logger | Token usage [answers pk=741] attempt 1 — in: 7729 (cached: 7168), out: 23, cost: $0.001827
2026-06-17 20:14:30 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=741] correct=1 wrong_fields=[]
2026-06-17 20:14:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=742:
{
  "questionID": 264,
  "participantID": 45,
  "answer_content_oriLAN": "De bibliotheek is een mooie voorziening, maar het zou geweldig zijn als er ook sociale activiteiten waren. Bijvoorbeeld een dagkaart om af en toe ergens heen te gaan, of uitnodigingen voor concerten, evenementen of gewoon samen een voetbalwedstrijd kijken. Dat zou helpen om mensen te ontmoeten en even uit het isolement te komen.\"",
  "initials": "S",
  "gender": "Man",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Turkse",
  "municipality": "Den Helder",
  "question_content": "Wat gaat er goed en welke obstakels erv


--- Evaluating answers pk=742 ---


2026-06-17 20:14:31 | INFO     | utils.token_logger | Token usage [answers pk=742] attempt 1 — in: 7778 (cached: 7168), out: 23, cost: $0.001888
2026-06-17 20:14:31 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=742] correct=1 wrong_fields=[]
2026-06-17 20:14:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=743:
{
  "questionID": 264,
  "participantID": 43,
  "answer_content_oriLAN": "Ik besteed mijn vrije tijd aan studeren, vooral aan taal. Maar ik heb niet genoeg materiaal. Meer taalleermiddelen of toegang tot online bronnen zou enorm helpen. En als we bijvoorbeeld een Museumkaart konden krijgen, zouden we ook via cultuur kunnen leren. Dat zou echt waardevol zijn.",
  "initials": "A",
  "gender": "Man",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Turks",
  "municipality": "Den Helder",
  "question_content": "Wat gaat er goed en welke obstakels ervaar je?"
}



--- Evaluating answers pk=743 ---


2026-06-17 20:14:32 | INFO     | utils.token_logger | Token usage [answers pk=743] attempt 1 — in: 7769 (cached: 7168), out: 23, cost: $0.001877
2026-06-17 20:14:32 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=743] correct=1 wrong_fields=[]


In [2]:
evaluator_version="initial_test"
notegroupID=11
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 730)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)


2026-06-17 20:28:28 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=730:
{
  "questionID": 264,
  "answer_content_oriLAN": "Municipality den Helder develops a new policy for newcomers and wants to know your feedback on this.Main indicators of interest:\nSafety 4\nOpvang 4\nHealth (mental health included) 4\nAccess to education 8\nAccess to work 9\nInspanningen\nOpvang",
  "question_content": "Wat gaat er goed en welke obstakels ervaar je?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=730 ---


2026-06-17 20:28:29 | INFO     | utils.token_logger | Token usage [answers pk=730] attempt 1 — in: 7725 (cached: 7168), out: 31, cost: $0.001902
2026-06-17 20:28:29 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=730] correct=0 wrong_fields=['answer_content_oriLAN', 'participantID']


In [3]:
evaluator_version="initial_test"
notegroupID=10
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in range(40, 42)]
    + [("questions", pk) for pk in [*range(244, 250), *range(256, 261)]]
    + [("answers", pk) for pk in [680, *range(704, 706)]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Reza: Note-taking form 29.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-17 20:35:26 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=40:
{
  "full_name": "Arsalan Azarmi",
  "initials": "Arsalan",
  "municipality": "Rotterdam"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating participants pk=40 ---


2026-06-17 20:35:27 | INFO     | utils.token_logger | Token usage [participants pk=40] attempt 1 — in: 9763 (cached: 0), out: 26, cost: $0.012464
2026-06-17 20:35:27 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=40] correct=0 wrong_fields=['municipality']
2026-06-17 20:35:27 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=41:
{
  "full_name": "Aida",
  "initials": "Aida"
}



--- Evaluating participants pk=41 ---


...

2026-06-17 20:35:41 | INFO     | utils.token_logger | Token usage [answers pk=680] attempt 1 — in: 9384 (cached: 8704), out: 23, cost: $0.002168
2026-06-17 20:35:41 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=680] correct=1 wrong_fields=[]
2026-06-17 20:35:41 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=704:
{
  "questionID": 260,
  "participantID": 41,
  "answer_content_oriLAN": "آیدا: با افزایش سطح حداقل درآمد انگیزه برای کار کردن تازه واردان بیشتر کنید. ارتقای سطح زبان و افزایش ارتباطات اجتماعی که باعث شود ترسشان از کار کردن کاهش یابد. هر شرکتی باید از افرادی ناتوانی جسمی دارند سهمیه دارند تا از این افراد استفاده کنند. برای کارفرمایان اجباری کنند که از افرادی که دوره اینبورخرینگ می گذاردند کار داوطلبانه کنند تا تجربه کارشان افزایش یابدو مالیات  پناهندگا و تازه وارد هستند را کمتر کنند. یا برای یک سال معافیت مالی بدهند. Increase motivation for newcomers to work by raising the minimum income level. Improve la


--- Evaluating answers pk=704 ---


2026-06-17 20:35:42 | INFO     | utils.token_logger | Token usage [answers pk=704] attempt 1 — in: 9555 (cached: 8832), out: 23, cost: $0.002238
2026-06-17 20:35:42 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=704] correct=1 wrong_fields=[]
2026-06-17 20:35:42 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=705:
{
  "questionID": 260,
  "participantID": 40,
  "answer_content_oriLAN": "ارسلان: حمایت اجتماعی از کسانی که تازه کار پیدا کردند قطع نشود. پیشنهاد کار داوطلبانه. واقعا از روز اول از کسی که وارد هلند می شود استفاده مثبت کنند. این فرد را آماده برای کار کنند. پناهجویان در طول پروسه رسیدگی و کسانی که دوره اینبورخرینگ را می گذارند در همین دوره. این فرصت طلایی است و انرژی فراوانی است. از روز اول بروید کار کنید. مهم نیست داوطلبانه باشد یا درآمدی. اینبورخرینگ واقعی زمانی رخ می دهد که بتوانی کار کنی. من با همسر و دختر خودم هم این توصیه را کردم. و آنان الان خوشحالند. همسرم با این که هیچ وقت کار نکرده بود و جرات نداش


--- Evaluating answers pk=705 ---


2026-06-17 20:35:44 | INFO     | utils.token_logger | Token usage [answers pk=705] attempt 1 — in: 9949 (cached: 8832), out: 23, cost: $0.002730
2026-06-17 20:35:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=705] correct=1 wrong_fields=[]


In [4]:
evaluator_version="initial_test"
notegroupID=23
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", pk) for pk in range(457, 465)]
    + [("answers", pk) for pk in [1088,1103]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading:  NOTITIES_Fatih (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/ NOTITIES_Fatih

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-17 20:42:14 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=457:
{
  "question_content": "Tijdens je inburgering, zijn er momenten geweest waarop je stress hebt ervaren? Waar kwam dat meestal door — bijvoorbeeld administratie, brieven, geldzaken, tijdsdruk of iets anders?\nWat doet de gemeente volgens jou goed om stress te verminderen, en wat zou beter kunnen? (Bijvoorbeeld: duidelijkere informatie, meer persoonlijke uitleg, minder brieven tegelijk...)",
  "main_indicator": [
    "safety",
    "stability",
    "health"
  ]
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating questions pk=457 ---


2026-06-17 20:42:16 | INFO     | utils.token_logger | Token usage [questions pk=457] attempt 1 — in: 16575 (cached: 0), out: 23, cost: $0.020949
2026-06-17 20:42:16 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=457] correct=1 wrong_fields=[]
2026-06-17 20:42:16 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=458:
{
  "question_content": "Heb je het gevoel dat je leven in Nederland op dit moment stabiel is? Wat helpt jou om stap voor stap een Een stabiel leven op te bouwen is belangrijk, maar wat maakt het soms moeilijk?\nAls je denkt aan een stabiel leven in Nederland, wat hoort daar volgens jou bij? (Bijvoorbeeld: vaste woning, werk, gezondheid, gezin, zekerheid over de toekomst.)",
  "main_indicator": [
    "safety",
    "stability",
    "health"
  ]
}



--- Evaluating questions pk=458 ---


...

2026-06-17 20:42:23 | INFO     | utils.token_logger | Token usage [questions pk=464] attempt 1 — in: 16523 (cached: 15488), out: 23, cost: $0.003460
2026-06-17 20:42:23 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=464] correct=1 wrong_fields=[]
2026-06-17 20:42:23 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=1088:
{
  "questionID": 448,
  "participantID": 76,
  "answer_content_oriLAN": "Ik ben wiskundeleraar van beroep en mijn diploma is in Nederland erkend. Dat is fijn, maar de taal is nog steeds een grote uitdaging voor mij. Daarom doe ik nu mee aan een project voor statushouders in het onderwijs. In dit project volg ik een taalcursus en loop ik twee dagen per week stage op een school. Zo kan ik veel Nederlands oefenen in de praktijk.\nIk heb dit project gevonden via een Turkse vriendin die hier al eerder aan meedeed. Er is geen baan­garantie, maar ik krijg wel Nederlandse werkervaring. Dat vind ik belangr


--- Evaluating answers pk=1088 ---


2026-06-17 20:42:25 | INFO     | utils.token_logger | Token usage [answers pk=1088] attempt 1 — in: 16293 (cached: 0), out: 23, cost: $0.020596
2026-06-17 20:42:25 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=1088] correct=1 wrong_fields=[]
2026-06-17 20:42:25 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=1103:
{
  "questionID": 457,
  "participantID": 75,
  "answer_content_oriLAN": "Ja, tijdens mijn inburgering had ik soms stress. Dat kwam vooral door de taal. Ik was bang dat mijn Nederlands niet snel genoeg beter werd. Ook moest ik veel formulieren en brieven begrijpen. Soms wist ik niet precies wat ik moest doen. Dat gaf druk en zorgde ervoor dat ik veel nadacht over mijn toekomst.\nDe gemeente helpt goed door duidelijk te antwoorden op vragen. Soms krijg ik snel een reactie, en dat geeft rust. Maar het kan beter als er minder wacht­tijd is en als er één vaste contactpersoon is. Dan hoef je niet steeds opnie


--- Evaluating answers pk=1103 ---


2026-06-17 20:42:26 | INFO     | utils.token_logger | Token usage [answers pk=1103] attempt 1 — in: 16284 (cached: 15488), out: 23, cost: $0.003161
2026-06-17 20:42:26 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=1103] correct=1 wrong_fields=[]


In [5]:
evaluator_version="initial_test"
notegroupID=13
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", pk) for pk in range(303, 309)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Data den Helder Ist session Ula (application/vnd.google-apps.document)


2026-06-17 20:45:59 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=303:
{
  "question_content": "Why did you choose this indicator? / Achieve? / Obstacles? / Actions taken? / What would help achieve this goal?",
  "main_indicator": [
    "language",
    "taal"
  ]
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data den Helder Ist session Ula

--- Skipping PARTICIPANT: no URL ---

--- Evaluating questions pk=303 ---


2026-06-17 20:46:00 | INFO     | utils.token_logger | Token usage [questions pk=303] attempt 1 — in: 3141 (cached: 0), out: 23, cost: $0.004156
2026-06-17 20:46:00 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=303] correct=1 wrong_fields=[]
2026-06-17 20:46:00 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=304:
{
  "question_content": "Why did you choose this indicator? / Achieve? / Obstacles? / Actions taken? / What would help achieve this goal?",
  "main_indicator": [
    "housing",
    "huisvesting & opvang"
  ]
}



--- Evaluating questions pk=304 ---


...

2026-06-17 20:46:03 | INFO     | utils.token_logger | Token usage [questions pk=306] attempt 1 — in: 3142 (cached: 2176), out: 26, cost: $0.001739
2026-06-17 20:46:03 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=306] correct=0 wrong_fields=['main_indicator']
2026-06-17 20:46:03 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=307:
{
  "question_content": "Why did you choose this indicator? / Achieve? / Obstacles? / Actions taken? / What would help achieve this goal?",
  "main_indicator": [
    "culture",
    "cultuur"
  ]
}



--- Evaluating questions pk=307 ---


2026-06-17 20:46:04 | INFO     | utils.token_logger | Token usage [questions pk=307] attempt 1 — in: 3141 (cached: 2176), out: 26, cost: $0.001738
2026-06-17 20:46:04 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=307] correct=0 wrong_fields=['main_indicator']
2026-06-17 20:46:04 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=308:
{
  "question_content": "Why did you choose this indicator? / Achieve? / Obstacles? / Actions taken? / What would help achieve this goal?",
  "main_indicator": [
    "education",
    "onderwijs"
  ]
}



--- Evaluating questions pk=308 ---


2026-06-17 20:46:05 | INFO     | utils.token_logger | Token usage [questions pk=308] attempt 1 — in: 3141 (cached: 2176), out: 26, cost: $0.001738
2026-06-17 20:46:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=308] correct=0 wrong_fields=['main_indicator']


In [6]:
evaluator_version="initial_test"
notegroupID=6
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 425)]
    + [("participants", 27)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Copy of Ale_ Note-taking form 28.11 (English translation) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-17 20:50:02 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=425:
{
  "questionID": 134,
  "participantID": 27,
  "answer_content_oriLAN": "Yes, because it’s not easy to manage study time, work, and integration. Integration is not only the courses but also the language, which is like studying a full university degree. It requires full-time attention.",
  "full_name": "Francielis Rivas",
  "initials": "Francielis",
  "learning_route": "B1-route",
  "place_of_origin": "Venezuela",
  "language_group": [
    "Spaans"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Amsterdam",
  "question_content": "Do the time requirements and obligations of the integration process affect your ability to search for or have work? If yes, in what way?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating answers pk=425 ---


2026-06-17 20:50:04 | INFO     | utils.token_logger | Token usage [answers pk=425] attempt 1 — in: 7404 (cached: 0), out: 23, cost: $0.009485
2026-06-17 20:50:04 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=425] correct=1 wrong_fields=[]
2026-06-17 20:50:04 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=27:
{
  "full_name": "Francielis Rivas",
  "initials": "Francielis",
  "learning_route": "B1-route",
  "place_of_origin": "Venezuela",
  "language_group": [
    "Spaans"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Amsterdam"
}



--- Evaluating participants pk=27 ---


2026-06-17 20:50:05 | INFO     | utils.token_logger | Token usage [participants pk=27] attempt 1 — in: 7771 (cached: 6656), out: 37, cost: $0.002596
2026-06-17 20:50:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=27] correct=0 wrong_fields=['learning_route', 'first_arrival_date', 'municipality', 'other_information']


In [7]:
evaluator_version="initial_test"
notegroupID=9
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", 36)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Nesrine: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-17 20:59:05 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=36:
{
  "full_name": "Ali Banat",
  "initials": "Al",
  "place_of_origin": "Syria",
  "municipality": "Den Haag"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating participants pk=36 ---


2026-06-17 20:59:06 | INFO     | utils.token_logger | Token usage [participants pk=36] attempt 1 — in: 9573 (cached: 0), out: 27, cost: $0.012236
2026-06-17 20:59:06 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=36] correct=0 wrong_fields=['place_of_origin']


In [8]:
evaluator_version="initial_test"
notegroupID=1
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", pk) for pk in range(1, 20)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx

--- Loading and extracting text from PARTICIPANT ---
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)


2026-06-17 21:01:08 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=1:
{
  "question_content": "How is your inburgering going so far?"
}


Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1

--- Evaluating questions pk=1 ---


2026-06-17 21:01:10 | INFO     | utils.token_logger | Token usage [questions pk=1] attempt 1 — in: 7191 (cached: 0), out: 23, cost: $0.009219
2026-06-17 21:01:10 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=1] correct=1 wrong_fields=[]
2026-06-17 21:01:10 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=2:
{
  "question_content": "Where do you feel pressure in your life, with the inburgering? What feels difficult to handle?"
}



--- Evaluating questions pk=2 ---


...

2026-06-17 21:01:25 | INFO     | utils.token_logger | Token usage [questions pk=17] attempt 1 — in: 7225 (cached: 6272), out: 23, cost: $0.002205
2026-06-17 21:01:25 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=17] correct=1 wrong_fields=[]
2026-06-17 21:01:25 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=18:
{
  "question_content": "Is there anything that we didn’t discuss in this session, that you feel is important to add?"
}



--- Evaluating questions pk=18 ---


2026-06-17 21:01:26 | INFO     | utils.token_logger | Token usage [questions pk=18] attempt 1 — in: 7201 (cached: 0), out: 23, cost: $0.009231
2026-06-17 21:01:26 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=18] correct=1 wrong_fields=[]
2026-06-17 21:01:26 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=19:
{
  "question_content": "Anything that is big and affects your life and needs attention from the municipality?"
}



--- Evaluating questions pk=19 ---


2026-06-17 21:01:27 | INFO     | utils.token_logger | Token usage [questions pk=19] attempt 1 — in: 7196 (cached: 6272), out: 23, cost: $0.002169
2026-06-17 21:01:27 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=19] correct=1 wrong_fields=[]


In [9]:
evaluator_version="initial_test"
notegroupID=4
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 273)]
    + [("participants", 23)]
    + [("questions", 64)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Fatih notes form 22 nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)


2026-06-17 21:05:10 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=273:
{
  "questionID": 64,
  "participantID": 24,
  "answer_content_oriLAN": "No, I’m not searching for work at all at the moment. I originally found my job through a direct online search, but because COA may relocate me again, I don’t want to invest time in a job that I might lose suddenly. For now, I’m focusing completely on inburgering and improving my Dutch, so I’m not using my previous job-search method.",
  "full_name": "Mehmet",
  "initials": "M",
  "question_content": "If you currently have a job:\n● Are you looking for new work?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling

--- Evaluating answers pk=273 ---


2026-06-17 21:05:11 | INFO     | utils.token_logger | Token usage [answers pk=273] attempt 1 — in: 15964 (cached: 0), out: 23, cost: $0.020185
2026-06-17 21:05:11 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=273] correct=1 wrong_fields=[]
2026-06-17 21:05:11 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=23:
{
  "full_name": "Özcan ikiz"
}



--- Evaluating participants pk=23 ---


2026-06-17 21:05:13 | INFO     | utils.token_logger | Token usage [participants pk=23] attempt 1 — in: 16247 (cached: 15232), out: 23, cost: $0.003403
2026-06-17 21:05:13 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=23] correct=1 wrong_fields=[]
2026-06-17 21:05:13 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=64:
{
  "question_content": "If you currently have a job:\n● Are you looking for new work?",
  "following_trigger": "currently have a job"
}



--- Evaluating questions pk=64 ---


2026-06-17 21:05:14 | INFO     | utils.token_logger | Token usage [questions pk=64] attempt 1 — in: 16304 (cached: 15232), out: 23, cost: $0.003474
2026-06-17 21:05:14 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=64] correct=1 wrong_fields=[]
